<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week05_1_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 수상작 리뷰

# 수상작 소개

**음악 장르 분류 AI 해커톤**  

(https://dacon.io/competitions/official/236056/codeshare/7528?page=1&dtype=recent)
(https://dacon.io/competitions/official/236056/codeshare/7525?page=1&dtype=recent) <- 다음주에 리뷰해볼것!

목표: 음악 장르를 분류하는 AI 알고리즘 개발

설명: 음악 샘플의 특징 정보로부터 음악의 장르를 분류하세요.  
(최근 음악 스트리밍 사이트 등에서 음악들을 자체적으로 분류하여 고객들에게 추천을 하거나,

수많은 음악들을 효과적으로 분류할 수 있는 방법들에 대한 수요와 시도가 지속적으로 증가하고 있습니다.

따라서 이러한 문제를 해결하는 첫걸음으로, 음악 샘플들의 대한 특징으로부터 장르를 자동적으로 분류하는 AI 모델을 생성하는 것입니다.)


평가: Macro F1 Score

데이터:
- ID : 음악 샘플 고유 ID
- 음악 샘플의 특징 정보
- genre : 음악 장르 (총 15개 종류)



---


## 코드 흐름 요약

- 부스팅(XGB, LGBM, CAT)
- 샘플 수 적은 클래스에만 오버샘플링


### 코드 흐름

1. 라이브러리 임포트
- 데이터 핸들링 툴
- 데이터 시각화 툴
- 머신러닝 모델, 교차검증 툴

2. 데이터 로드
- 무의미한 변수 제거(ID)
- 기초 통계량 확인
- 결측치 확인: 결측치 없음
- 변수 의미 확인
- 숫자형/범주형 칼럼별 나누어 변수에 배정

3. EDA
- 목표 피처 분포 확인(countplot)
- 양적 변수 분포 확인
  - sns.histplot + 왜도 계산
  - sns.boxplot


4. 데이터 전처리
- 범주형 변수: 라벨 인코딩
- 숫자형 변수: 정규화, MinMaxScaler()
- 함수 정의


5. 모델링
- 교차검증(KFold)  
* 오버샘플링은 샘플 수가 가장 적은 클래스만 적용, 그 다음으로 적은 클래스의 샘플 수로 맞춰주었음.  
(다른 클래스에도 오버샘플링을 적용한 경우엔 성능이 오히려 감소함. )


단일 모델
- XGBClassifier
- LGBMClassifier
- CatBoostClassifier

앙상블
- Hard Voting





## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점 등 정리 및 주요 코드 3줄 이상 작성

- 숫자형/범주형 칼럼별 지정, 저장해두면 시각화, 인코딩 모두 편리하게 코드 작성 가능





In [1]:
'''
numeric_columns = df.columns[(df.dtypes == int) | (df.dtypes == float)]
categorical_columns = df.columns[(df.dtypes == 'object')]
'''

"\nnumeric_columns = df.columns[(df.dtypes == int) | (df.dtypes == float)]\ncategorical_columns = df.columns[(df.dtypes == 'object')]\n"

함수 지정 후 값 return만 받아 코드 직관화


In [2]:
# Hard Voting 함수 정의
'''
def hard_voting(all_preds: List[NDArray[np.int]]) -> List[int]:
    preds = []
    for all_pred in all_preds:
        p = stats.mode(all_pred).mode.item()
        preds.append(p)
    return preds
'''

'\ndef hard_voting(all_preds: List[NDArray[np.int]]) -> List[int]:\n    preds = []\n    for all_pred in all_preds:\n        p = stats.mode(all_pred).mode.item()\n        preds.append(p)\n    return preds\n'

In [4]:
'''
all_test_preds = np.concatenate(all_test_preds, axis=-1)

# 하드보팅 결과값을 바로 제출 파일에 지정
submission['genre'] = label_encoder.inverse_transform(hard_voting(all_test_preds))
submission.to_csv('./data/submission.csv', index=False)
'''

"\nall_test_preds = np.concatenate(all_test_preds, axis=-1)\n\n# 하드보팅 결과값을 바로 제출 파일에 지정\nsubmission['genre'] = label_encoder.inverse_transform(hard_voting(all_test_preds))  \nsubmission.to_csv('./data/submission.csv', index=False)  \n"

다만 함수를 엄청나게 많이 정의했는데 모든 함수의 의미를 따라가기 넘 어려웠다 ...  
주석이 있어야 공유 용이할듯  
코드 작성하는 사람 입장에서는 아주 편리해보이긴 하다